In [32]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

import pandas as pd
import numpy as np
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import itertools
from prophet.diagnostics import cross_validation,performance_metrics
from prophet.plot import plot_cross_validation_metric

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [2]:
# Create a continent mapping
continent_map = {
    'Africa': ['Algeria', 'Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cameroon', 'Central African Republic', 'Chad', 'Comoros', 'Congo', 'Cote d\'Ivoire', 'Democratic Republic of the Congo', 'Djibouti', 'Egypt', 'Equatorial Guinea', 'Eritrea', 'Eswatini', 'Ethiopia', 'Gabon', 'Gambia', 'Ghana', 'Guinea', 'Guinea-Bissau', 'Kenya', 'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Mali', 'Mauritania', 'Mauritius', 'Morocco', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 'Rwanda', 'Sao Tome and Principe', 'Senegal', 'Seychelles', 'Sierra Leone', 'Somalia', 'South Africa', 'South Sudan', 'Sudan', 'Tanzania', 'Togo', 'Tunisia', 'Uganda', 'Zambia', 'Zimbabwe'],
    'Asia': ['Afghanistan', 'Armenia', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Bhutan', 'Brunei Darussalam', 'Cambodia', 'China', 'Cyprus', 'Georgia', 'India', 'Indonesia', 'Iran', 'Iraq', 'Israel', 'Japan', 'Jordan', 'Kazakhstan', 'Kuwait', 'Kyrgyzstan', 'Laos', 'Lebanon', 'Malaysia','Maldives', 'Mongolia', 'Myanmar', 'Nepal', 'North Korea', 'Oman', 'Pakistan', 'Palestine', 'Philippines', 'Qatar', 'Saudi Arabia', 'Singapore', 'South Korea', 'Sri Lanka', 'Syria', 'Taiwan', 'Tajikistan', 'Thailand', 'Timor-Leste', 'Turkey', 'Turkmenistan', 'United Arab Emirates', 'Uzbekistan', 'Vietnam', 'Yemen'],
    'Europe': ['Albania', 'Andorra', 'Austria', 'Belarus', 'Belgium', 'Bosnia and Herzegovina', 'Bulgaria', 'Croatia', 'Czech Republic', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Kosovo', 'Latvia', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Malta', 'Moldova', 'Monaco', 'Montenegro', 'Netherlands', 'North Macedonia', 'Norway', 'Poland', 'Portugal', 'Romania', 'Russia', 'San Marino', 'Serbia', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Ukraine', 'United Kingdom', 'Vatican City'],
    'North America': ['Canada', 'Greenland', 'Mexico', 'United States'],
    'Oceania': ['Australia', 'Fiji', 'New Zealand', 'Papua New Guinea'],
    'South America': ['Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Ecuador', 'Guyana', 'Paraguay', 'Peru', 'Suriname', 'Uruguay', 'Venezuela']
}


In [95]:
dataset = pd.read_csv ("World Energy Consumption.csv")

In [ ]:
col_electricity = ['low_carbon_electricity',
            'nuclear_electricity',
            'oil_electricity',
            'other_renewable_electricity',
            'other_renewable_exc_biofuel_electricity',
            'solar_electricity',
            'wind_electricity',
            'gas_electricity',
            'hydro_electricity',
            'biofuel_electricity',
            'coal_electricity']
col_cons =['nuclear_consumption',
           'oil_consumption',
           'solar_consumption',
           'wind_consumption',
           'fossil_fuel_consumption',
           'gas_consumption',
           'hydro_consumption',
           'biofuel_consumption',
           'coal_consumption',
           'low_carbon_consumption',]
rename_columns = {
    'nuclear_electricity': 'nuclear',
    'oil_electricity': 'petroleo',
    'solar_electricity': 'solar',
    'wind_electricity': 'eolica',
    'fossil_electricity': 'fosil',
    'gas_electricity': 'gas',
    'hydro_electricity': 'hidro',
    'biofuel_electricity': 'bio fuel',
    }
rename_columns_cons = {
    'nuclear_consumption': 'nuclear',
    'oil_consumption': 'petroleo',
    'solar_consumption': 'solar',
    'wind_consumption': 'eolica',
    'fossil_fuel_consumption': 'fosil',
    'gas_consumption': 'gas',
    'hydro_consumption': 'hidro',
    'biofuel_consumption': 'bio fuel',
    }


In [ ]:
'''mundial = dataset.groupby('year')[col_electricity].sum().reset_index()
mundial['y'] = mundial.sum(axis=1)
mundial.drop(columns=col_electricity, inplace=True)
mundial['ds'] = mundial['year']
mundial.drop(columns=['year'], inplace=True)
mundial = mundial.groupby('ds').sum().reset_index()
mundial


   #df_grouped['carbon'] = df_grouped['low_carbon_electricity'] + df_grouped['coal_electricity']
    #df_grouped['renovable'] = (df_grouped['other_renewable_electricity'] +
     #                      df_grouped['other_renewable_exc_biofuel_electricity'])
        #mundial.drop(columns=['carbon', 'renovable'], inplace=True)
        '''

,ds,y
0,1900,1900.000
1,1901,1901.000
2,1902,1902.000
3,1903,1903.000
4,1904,1904.000
...,...,...
118,2018,461859.069
119,2019,467083.849
120,2020,464489.305
121,2021,490521.103


In [ ]:
def prophet_energias(dataset):
    prophet_df = pd.DataFrame()
    columns = ['nuclear', 'petroleo', 'solar', 'eolica', 'gas',
       'hidro', 'bio fuel', 'carbon']

    # Datos mundiales
    mundial = dataset.groupby('year')[col_electricity].sum().reset_index()
    mundial['y'] = mundial.sum(axis=1)
    mundial.drop(columns=col_electricity, inplace=True)
    mundial['ds'] = mundial['year']
    mundial.drop(columns=['year'], inplace=True)
    mundial = mundial.groupby('ds').sum().reset_index()

    # generacion datos de produccion ordenandos
    prueba = dataset.groupby('year')[col_electricity].sum()
    prueba = prueba[prueba.index>1985].reset_index()
    prueba['carbon'] = prueba['low_carbon_electricity'] + prueba['coal_electricity']
    prueba['renovable']  = prueba['other_renewable_electricity'] + prueba['other_renewable_exc_biofuel_electricity']
    prueba.drop(columns=['low_carbon_electricity','other_renewable_electricity','other_renewable_exc_biofuel_electricity','coal_electricity'],inplace=True)
    prueba.rename(columns=rename_columns,inplace=True)

    # generacion datos de consumo ordenandos
    prueba_consumo = dataset.groupby('year')[col_cons].sum()
    prueba_consumo = prueba_consumo[prueba_consumo.index>1985].reset_index()
    prueba_consumo['carbon'] = prueba_consumo['low_carbon_consumption'] + prueba_consumo['coal_consumption']
    prueba_consumo.drop(columns=['low_carbon_consumption','coal_consumption'],inplace=True)
    prueba_consumo.rename(columns=rename_columns_cons,inplace=True)

    fig = go.Figure() #grafico mundial
    fig1 = go.Figure() #grafico de energia

    target_year = 2030
    # modelo predictivo mundial y grafico
    p = dataset.groupby('year')[col_cons].sum().sum(axis=1).to_frame('consumo').reset_index()
    mundial['consumo'] = p['consumo']
    mundial = mundial[mundial['ds']>=1985]
    mundial['ds'] = pd.to_datetime(mundial['ds'], format='%Y')

    changepoint_prior_scale = 0.1
    seasonality_prior_scale= 1.0
    seasonality_mode = 'additive'
    model = Prophet(
        yearly_seasonality=True,
        changepoint_prior_scale=changepoint_prior_scale,
        seasonality_prior_scale=seasonality_prior_scale,
        seasonality_mode=seasonality_mode
    )
    prophet_df = mundial.copy()

    model.add_regressor('consumo')
    model.fit(prophet_df)

    last_date = prophet_df['ds'].max()
    years_to_predict = target_year - last_date.year

    future = model.make_future_dataframe(periods=years_to_predict, freq='Y')
    future = future.merge(
        prophet_df[['ds', 'consumo']],
        on='ds',
        how='left'
    )
    # Completar valores faltantes usando forward fill (último valor conocido)
    future['consumo'] = future['consumo'].fillna(method='ffill')

    forecast = model.predict(future)

    fig.add_trace(go.Scatter(
        x=prophet_df['ds'],
        y=prophet_df['y'],
        mode='lines+markers',
        name='Datos Historicos',
        line=dict(color='red', width=3)
    ))
    future_forecast = forecast[forecast['ds'] >= last_date]

    if not future_forecast.empty:
        # Añadir línea que conecta los datos reales con la predicción
        fig.add_trace(go.Scatter(
            x=[prophet_df['ds'].iloc[-1], future_forecast['ds'].iloc[0]],
            y=[prophet_df['y'].iloc[-1], future_forecast['yhat'].iloc[0]],
            mode='lines',
            line=dict(color='red', width=0.5),
            showlegend=False
        ))

        fig.add_trace(go.Scatter(
            x=future_forecast['ds'],
            y=future_forecast['yhat'],
            mode='lines+markers',
            name='Predicción',
            line=dict(color='blue', width=3)
        ))
    prophet_df = pd.DataFrame()
    # modelo por energia y impresion de graficos
    prophet_df['ds'] = pd.to_datetime(prueba['year'], format='%Y')
    for c in columns:
        prophet_df['y'] = prueba[c]
        prophet_df['consumo'] = prueba_consumo[c]

    # Crear nuevo modelo en cada iteración
        model = Prophet(
            yearly_seasonality=True,
            changepoint_prior_scale=0.01,
            seasonality_prior_scale=1.0,
            seasonality_mode='additive')

        model.add_regressor('consumo')
        model.fit(prophet_df)

        last_date = prophet_df['ds'].max()
        years_to_predict = target_year - last_date.year

        future = model.make_future_dataframe(periods=years_to_predict, freq='Y')
        future = future.merge(
            prophet_df[['ds', 'consumo']],
            on='ds',
            how='left')

        # Completar valores faltantes usando forward fill (último valor conocido)
        future['consumo'] = future['consumo'].fillna(method='ffill')

        forecast = model.predict(future)

        fig1.add_trace(go.Scatter(
            x=prophet_df['ds'],
            y=prophet_df['y'],
            mode='lines+markers',
            name= f'{c}-Histórico',
            line=dict(color="sandybrown", width=0.5)
        ))
        future_forecast = forecast[forecast['ds'] >= last_date]

        if not future_forecast.empty:
         # Añadir línea que conecta los datos reales con la predicción
            fig1.add_trace(go.Scatter(
                x=[prophet_df['ds'].iloc[-1], future_forecast['ds'].iloc[0]],
                y=[prophet_df['y'].iloc[-1], future_forecast['yhat'].iloc[0]],
                mode='lines',
                line=dict(color='red', dash='dash', width=1.5),
                showlegend=False
            ))
            fig1.add_trace(go.Scatter(
                x=future_forecast['ds'],
                y=future_forecast['yhat'],
                mode='lines',
                name=f'{c}-Predicción',
                line=dict(color='green', width=0.5)
            ))
        fig1.update_layout(
            title="Predicciones de Energías",
            xaxis=dict(title="Fecha", showgrid=True, griddash='dot'),
            yaxis=dict(title="Valor", showgrid=True, griddash='dot'),
            showlegend=False,
            width=700,
            height=600,
            template="plotly_white"
             )
        fig.update_layout(
            title="Predicciones de Energías",
            xaxis=dict(title="Fecha", showgrid=True, griddash='dot'),
            yaxis=dict(title="Valor", showgrid=True, griddash='dot'),
            showlegend=False,
            width=700,
            height=600,
            template="plotly_white"
             )

    fig.show()
    fig1.show()

In [130]:
prophet_energias(dataset)

20:27:39 - cmdstanpy - INFO - Chain [1] start processing
20:27:39 - cmdstanpy - INFO - Chain [1] done processing
20:27:39 - cmdstanpy - INFO - Chain [1] start processing
20:27:39 - cmdstanpy - INFO - Chain [1] done processing
20:27:39 - cmdstanpy - INFO - Chain [1] start processing
20:27:40 - cmdstanpy - INFO - Chain [1] done processing
20:27:40 - cmdstanpy - INFO - Chain [1] start processing
20:27:40 - cmdstanpy - INFO - Chain [1] done processing
20:27:40 - cmdstanpy - INFO - Chain [1] start processing
20:27:40 - cmdstanpy - INFO - Chain [1] done processing
20:27:40 - cmdstanpy - INFO - Chain [1] start processing
20:27:40 - cmdstanpy - INFO - Chain [1] done processing
20:27:40 - cmdstanpy - INFO - Chain [1] start processing
20:27:40 - cmdstanpy - INFO - Chain [1] done processing
20:27:40 - cmdstanpy - INFO - Chain [1] start processing
20:27:41 - cmdstanpy - INFO - Chain [1] done processing
20:27:41 - cmdstanpy - INFO - Chain [1] start processing
20:27:41 - cmdstanpy - INFO - Chain [1]